# Module 14: Categorical Variables and Interactions

We continue with the European hotels case. Categorical coefficients always
compare groups to a reference category; interactions allow a slope to differ
across groups. Prediction tables and plots make both ideas easier to explain.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import statsmodels.formula.api as smf

sns.set_theme(style="whitegrid")

In [ ]:
hotels = pd.read_csv("../../data/raw/hotels_course.csv")
hotels.groupby(["city", "accommodation"]).size().unstack(fill_value=0)

## 1. A categorical predictor is a set of comparisons

In [ ]:
vienna = hotels.query("city == 'Vienna'").copy()
category_model = smf.ols(
    "log_price ~ distance_miles + rating + C(stars) + C(accommodation)",
    data=vienna,
).fit(cov_type="HC3")
category_model.summary()

In [ ]:
category_terms = category_model.params.index[category_model.params.index.str.startswith("C(")]
pd.DataFrame(
    {"estimate": category_model.params[category_terms], "robust_se": category_model.bse[category_terms]}
).round(3)

Statsmodels uses the alphabetically first observed category as the reference
unless we specify another. The intercept therefore describes a particular
reference hotel, not an average hotel.

In [ ]:
comparison_hotels = pd.DataFrame(
    {
        "distance_miles": [2.0, 2.0],
        "rating": [4.0, 4.0],
        "stars": [4.0, 4.0],
        "accommodation": ["Apartment", "Hotel"],
    }
)
comparison_hotels["predicted_price"] = np.exp(category_model.predict(comparison_hotels))
comparison_hotels.round(1)

## 2. Interactions let the distance slope differ

In [ ]:
interaction_model = smf.ols(
    "log_price ~ distance_miles * C(accommodation) + rating + C(stars)",
    data=vienna,
).fit(cov_type="HC3")
interaction_model.summary()

In [ ]:
base_slope = interaction_model.params["distance_miles"]
difference = interaction_model.params["distance_miles:C(accommodation)[T.Hotel]"]
slopes = pd.Series(
    {"Apartment": base_slope, "Hotel": base_slope + difference}, name="log-price slope"
)
slopes.round(3)

In [ ]:
grid = pd.DataFrame(
    {
        "distance_miles": np.tile(np.linspace(0.2, 7, 50), 2),
        "accommodation": np.repeat(["Apartment", "Hotel"], 50),
        "rating": 4.0,
        "stars": 4.0,
    }
)
grid["predicted_price"] = np.exp(interaction_model.predict(grid))

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
sns.lineplot(data=grid, x="distance_miles", y="predicted_price", hue="accommodation", ax=ax)
ax.set(title="Interaction shown as different slopes", xlabel="Distance (miles)", ylabel="Predicted price")
plt.show()

## 3. City effects and model comparison

In [ ]:
pooled = smf.ols(
    "log_price ~ distance_miles + rating + C(stars) + C(accommodation) + C(city)",
    data=hotels,
).fit(cov_type="HC3")
pooled_interaction = smf.ols(
    "log_price ~ distance_miles * C(city) + rating + C(stars) + C(accommodation)",
    data=hotels,
).fit(cov_type="HC3")

In [ ]:
pd.DataFrame(
    {
        "model": ["common distance slope", "city-specific slopes"],
        "adjusted_r2": [pooled.rsquared_adj, pooled_interaction.rsquared_adj],
        "aic": [pooled.aic, pooled_interaction.aic],
    }
).round(2)

**Interpretation checkpoint:** An interaction coefficient is a *difference in
slopes*. The clearest explanation usually combines the relevant coefficients
or shows predicted values, as we did above.

Continue to Module 15 before running the Session 2 reconstruction exercise.